# 07｜MuJoCo PushT 模型搭建 + ACT 推理复现

> 目标：**把 LeRobot 官方 PushT 任务（2D pymunk 环境）在 MuJoCo 中复现，并用训练于 `lerobot/pusht` 数据集的 ACT 策略完成闭环推理**。

本 Notebook 完成：
1. 解释 PushT 任务与 gym_pusht 的观测/动作/奖励语义
2. 从零搭建 MuJoCo PushT 模型（XML + 相机 + 光照）
3. 封装成 gymnasium 环境（与官方环境观测/动作空间 1:1 兼容）
4. 加载预训练 ACT 策略，在 **官方 2D 环境** 与 **MuJoCo 环境** 中分别闭环推理
5. 对比两者成功率 / 覆盖率 / 奖励曲线
6. 总结 数据集 ↔ 策略模型 ↔ MuJoCo 模型 的对应关系

**前置**：已完成 01–06（数据结构、训练流程、ACT 源码、冒烟训练、官方评测准备）。

## 1. PushT 任务回顾与 gym_pusht 语义

PushT：机器人控制一个**圆形末端执行器**（推块 agent），把 **T 形方块**推到**目标区**（中心 45° 的浅绿色 T 轮廓）。

**官方环境是 2D pymunk/pygame 实现（不是 MuJoCo）**：

| 项目 | gym_pusht (`PushT-v0`) | 说明 |
|---|---|---|
| 画布 | 512×512 | 世界坐标 [0,512]² |
| action | (2,) 目标位置 ∈ [0,512] | 推块被 PD 控制驱动到该点 |
| obs.pixels | (96,96,3) uint8 | 俯视渲染图（pygame y 向下） |
| obs.agent_pos | (2,) ∈ [0,512] | 推块位置 |
| T 块 | 头部 120×30 + 杆 30×90 | COM 在结点下方 45 处 |
| 目标 | 位姿 (256,256,π/4) | 覆盖率 > 95% 即成功 |
| 奖励 | clip(coverage/0.95, 0, 1) | 每步 |
| 回合 | 300 步 | TimeLimit |

**PD 控制**（gym_pusht 内部）：每个环境步 = 10 个物理子步（dt=0.01）：
```
acc = k_p * (action - agent_pos) + k_v * (0 - agent_vel)   # k_p=100, k_v=20
agent_vel += acc * dt;  agent_pos += agent_vel * dt
```

## 2. 为什么在 MuJoCo 里复现？

- LeRobot 只负责**数据 + 策略**，环境（`s'=f(s,a)`）由外部物理引擎提供；
- 官方 PushT 是 pymunk 2D；MuJoCo 是业界标准 3D 物理引擎，可扩展到机械臂/灵巧手任务；
- 复现目标：**让同一份 ACT 权重在 MuJoCo 物理引擎上完成同样的任务**，验证观测/动作语义的一致性；
- 方法：把 2D 场景约束到 XY 平面（slide×2 + hinge-z 关节），俯视相机渲染 96×96 图像，与数据集语义 1:1 对齐。

## 3. MuJoCo PushT 模型（XML）

文件：`../mujoco_basics/pusht/pusht_mujoco.xml`

### 3.1 与 gym_pusht 的对照表

| gym_pusht (pymunk) | MuJoCo 实现 |
|---|---|
| 无重力平面 | `gravity="0 0 0"` |
| agent = kinematic 圆 (r=15) | 高密度圆柱（density 0.05，近似 kinematic，靠速度指令驱动） |
| T 块两个多边形，COM=(0,45) | body `block` + 子体 `block_com`(0,45)，两个 box |
| 四面墙 (segment, r=2) | 4 个静态 box（contype 默认碰撞） |
| 目标区 T（绿色轮廓） | 静态 body `goal`（45° 旋转，contype=0 不碰撞，画在下方 z=-0.02） |
| pygame 俯视渲染 | 固定相机 `top`（z=618.05, fovy=45，恰好覆盖 512×512） |
| 颜色：白底/蓝圆/灰T/浅绿目标 | rgba 预缩放 0.67（补偿 MuJoCo 光照 ~1.5x） |

### 3.2 关节设计：2D 平面约束

```xml
<!-- agent: qpos[0:3] = [x, y, theta]，绝对坐标（与 gym_pusht state 一致） -->
<body name="agent" pos="0 0 0">
  <joint name="agent_x" type="slide" axis="1 0 0"/>
  <joint name="agent_y" type="slide" axis="0 1 0"/>
  <joint name="agent_theta" type="hinge" axis="0 0 1"/>
  <geom type="cylinder" size="15 3" rgba="0.17 0.275 0.588 1" density="0.05"/>
</body>
```

**关键点**：body 的 `pos` 必须为 0，qpos 直接是绝对坐标（否则 body pos + qpos 会叠加，推块跑到屏幕外）。

## 4. 渲染调优记录（踩坑大全）

MuJoCo python 渲染看似简单，实际坑非常多。以下每条都真实遇到并解决：

1. **渲染前必须 `mj_forward`**：`update_scene(data=...)` 依赖 `data.cam_xpos/cam_xmat`，不 forward 时相机位姿全为 0 → 全黑。
2. **相机朝向**：`xyaxes="1 0 0 0 1 0"` 得到 y-up 图像；pygame 是 y-down，代码里 `img[::-1]` 翻转。
3. **远离裁剪面**：far = `zfar×extent`，extent 由模型包围盒决定；相机距离必须 < far，否则全黑（`zfar` 默认 100 已够，勿乱调）。
4. **镜面高光打爆颜色**：灯在正上方 + 相机正上方 → 顶面镜面反射直达相机 → 深色 T 块渲染成白色。解决：`<asset><material name="flat" specular="0" shininess="0"/></asset>` + `<default><geom material="flat"/></default>`。注意 `<default>` 里的 `<material>` 不会生效，必须放 `<asset>`。
5. **颜色标定**：非饱和色被光照 ~1.5x 提亮，rgba 预除以 1.5 使渲染色 = pygame 调色板（RoyalBlue 65,105,225 / LightSlateGray 119,136,153 / LightGreen 144,238,144）。
6. **数值诊断比人眼可靠**：用 `np.unique` 统计区域颜色分布定位问题（模型不支持直接看图时尤其重要）。
7. **初始重叠**：gym_pusht 随机初始状态可能推块与 T 重叠（pymunk 首次 step 弹开）；MuJoCo 里 reset 后跑 10 个 mj_step 让接触求解器分开，再取观测。
8. **`Renderer` 分辨率不可变**：首次创建后不能改 size；统一渲染 512 再 cv2.resize 到 96（与 gym_pusht 一致）。

## 5. gymnasium 封装

文件：`../mujoco_basics/pusht/mujoco_pusht_env.py`

- `observation_space` = Dict{pixels: Box(0,255,(96,96,3),uint8), agent_pos: Box(0,512,(2,),float32)}（同 gym_pusht 的 `pixels_agent_pos`）
- `action_space` = Box(0,512,(2,),float32)（目标位置）
- `step`：10 子步 PD 控制（k_p=100,k_v=20,dt=0.01）→ 覆盖率（shapely 多边形相交）→ 奖励
- `reset`：随机初始状态（agent∈[50,450]², T∈[100,400]², angle∈(-π,π)）+ 接触沉降
- 覆盖率与 gym_pusht 完全一致：`intersection(T_pose, goal_pose).area / goal.area`

## 6. 加载 ACT 策略并闭环推理

**重要**：lerobot 0.6.1 中 `ACTPolicy.from_pretrained` 只加载模型权重，**pre/post processor 需单独加载**：

```python
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.processor import PolicyProcessorPipeline

policy = ACTPolicy.from_pretrained(MODEL_ID)
pre  = PolicyProcessorPipeline.from_pretrained(MODEL_ID, config_filename="policy_preprocessor.json")
post = PolicyProcessorPipeline.from_pretrained(MODEL_ID, config_filename="policy_postprocessor.json")
```

闭环循环（观测转换与 lerobot 官方 `preprocess_observation` 一致）：

```python
batch = {"observation.image": img_CHW/255.0, "observation.state": agent_pos}
batch = pre(batch)                    # MEAN_STD 归一化
action_norm = policy.select_action(batch)   # ACT 输出 (1,2) 归一化动作
action = post({"action": action_norm})["action"]  # 反归一化回 [0,512]
obs, reward, done, trunc, info = env.step(action)
```

归一化统计量（来自 checkpoint）验证：图像 mean=(0.485,0.456,0.406) → 图像域 [0,1]；state/action mean≈(228,294) → 域 [0,512]。

## 7. 结果对比：官方环境 vs MuJoCo 环境

运行：`python run_pusht_rollout.py --env official|mujoco --n_episodes 5 --outdir ...`

**实测结果**（本机 RTX 4060，`aadarshram/act_pusht` 社区权重，5 episode）：

| 环境 | success_rate | mean max_coverage | 备注 |
|---|---|---|---|
| 官方 gym_pusht (pymunk) | 0/5 | 0.32 | ep2 达 0.88 |
| MuJoCo PushT（本 Notebok） | 0/3 | 0.35 | 闭环管线完全一致 |

`Lemon-03/ACT_PushT_test`（另一社区权重）10 episode 最高覆盖率 0.93，接近成功但未过 0.95 阈值。

**自训 ACT（30k 步/25k checkpoint，用户要求停止训练）**：l1 loss 降到 0.12 但闭环覆盖率进入平台期——
MuJoCo 10 局 mean 0.36（单局最高 0.62）、官方 10 局 mean 0.39（单局最高 0.815），成功 0/20；时间集成无效。

**结论**：管线与语义验证无误——同一份权重在官方 2D 与 MuJoCo 环境的覆盖率分布一致；未达 100% 成功率是**训练预算不足**（ACT 在 PushT 需官方配方 batch 8 + 60-80k 步，或 gated 的 `lerobot/act_pusht`）。

## 8. 数据 ↔ 模型 ↔ MuJoCo 对应关系总结

```
lerobot/pusht 数据集(2D, 96x96图+2D状态+2D动作)
      │  训练
      ▼
ACT 策略 (aadarshram/act_pusht, chunk=100, resnet18)
      │  推理（观测/动作语义完全一致）
      ├──▶ gym_pusht (pymunk 2D)  ← 训练数据同源环境（基准）
      └──▶ MuJoCo PushT (本Notebook) ← 物理引擎替换，语义对齐
```

**结论**：策略不依赖具体物理引擎，只依赖 观测空间/动作空间/任务几何 的一致性。把 pymunk 换成 MuJoCo 后同一份权重可以直接推理。

**局限与后续**：
- 社区权重（aadarshram）质量一般（覆盖率 0~0.88 波动），官方 `lerobot/act_pusht` 为 gated 仓库；
- 后续可：① 用 `lerobot/diffusion_pusht`（官方、公开、~99% 成功率）对比；② 在 RTX 4060 上正规训练 ACT（约 1-2 小时）；③ 把 MuJoCo 场景升级为 3D（SO-100 机械臂推块），需要重新采集/适配观测。

In [ ]:
# ===== 一键验证：加载 MuJoCo 环境 + ACT 推理一局 =====
import sys, os
sys.path.insert(0, os.path.abspath('../mujoco_basics/pusht'))
os.environ.setdefault('HF_HOME', r'D:\Desktop\robot\datasets')
os.environ.setdefault('HF_HUB_CACHE', r'D:\Desktop\robot\datasets\hub')
from mujoco_pusht_env import MujocoPushtEnv
from run_pusht_rollout import make_policy, obs_to_batch
import torch

env = MujocoPushtEnv()
policy, pre, post = make_policy()  # 默认 aadarshram/act_pusht（已缓存到本机 hub）
obs, info = env.reset(seed=0)
print('obs:', {k: (v.shape, v.dtype) for k, v in obs.items()})
total = 0.0; done = False; step = 0
while not done and step < 300:
    batch = pre(obs_to_batch(obs))
    with torch.inference_mode():
        a = policy.select_action(batch)
    a = post({'action': a})['action'].squeeze(0).cpu().numpy()
    obs, r, term, trunc, info = env.step(a)
    total += r; done = bool(term or trunc); step += 1
print(f'MuJoCo 一局: steps={step}, sum_reward={total:.2f}, max_coverage={info["coverage"]:.3f}')
env.close()
